# Reading & Writing Files 📁

Two things have quietly bugged us so far:

- In the **chatbot** lesson, the conversation lived in a list that **died** the moment the program ended.
- In **structured output**, the LLM's lovely JSON vanished as soon as the cell finished running.

Right now, every brilliant thing your AI produces disappears when you close the notebook — like a dream you forgot by breakfast. 😴

Files fix that. A file is just data saved to disk that **survives** after your program stops. By the end of this lesson you'll read documents *into* your AI and save its answers *out* to disk — the bridge between a toy and a real app.

## 1. Writing a file

Let's start by writing, because it also *creates* the file we'll read next. The pattern:

```python
with open("filename", "w", encoding="utf-8") as f:
    f.write("some text")
```

- `"w"` = **write mode** (creates the file, or overwrites it if it already exists).
- `encoding="utf-8"` = handles emojis and accented characters cleanly. **Always include it** — it saves you from weird errors on Windows.

In [1]:
with open("note.txt", "w", encoding="utf-8") as f:
    f.write("My memory is so bad.\n")
    f.write("How bad is it?\n")
    f.write("How bad is what?\n")

print("Saved! Open note.txt to read the joke.")

Saved! Open note.txt to read the joke.


## 2. Reading a file

Same pattern, but with `"r"` (**read mode**). `.read()` pulls the whole file in as one string:

In [2]:
with open("note.txt", "r", encoding="utf-8") as f:
    content = f.read()

print(content)

My memory is so bad.
How bad is it?
How bad is what?



### What's that `with` about?

`with` is a safety net: it **automatically closes the file** when the block ends — even if something crashes midway. Think of it as the well-mannered guest who shuts the door behind them. 🚪

Without `with`, you'd have to remember `f.close()` yourself — and forgetting it can mean lost data or locked files. So: always use `with`.

## 3. The payoff: read a document → ask the LLM about it

This is *why* files matter for AI. Real documents live on disk — meeting notes, contracts, reports. The workflow is simple:

**read the file → put its text in the prompt → ask the LLM.**

We already have a file called `meeting.txt` in this folder. It has a short meeting transcript.

Let's read it, and then ask the LLM to summarize it.

In [7]:
# read the document
with open("meeting.txt", "r", encoding="utf-8") as f:
    document = f.read()
    
print(document)    


Production Meeting - May 29

Ramesh (Production): Last week we made 4,200 units, but our target was 5,000. Machine 3 was down for two days, so we fell behind.
Anjali (Maintenance): Machine 3 is working again now. I will check all the machines this week so we do not get sudden breakdowns.
Vikram (Quality): We found 80 units with paint problems. The paint is not drying properly. We must fix the painting step.
Suresh (Store): The steel supply is late again. The new stock will arrive on Thursday.
Meena (Manager): Okay. Ramesh, please plan some extra shifts to cover the gap. Anjali, finish the machine check by Friday. Vikram, send me a short report on the paint problem by tomorrow.



In [8]:
from dotenv import load_dotenv
from google import genai

load_dotenv()
client = genai.Client()

# put it in the prompt and ask
prompt = f"""
Here is a meeting transcript:
{document}

Summarize it in 3 short bullet points.
"""

try:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    summary = response.text
except Exception as e:
    summary = f"Something went wrong: {e}"

print(summary)

Here's a summary in 3 short bullet points:

*   Production missed its 5,000-unit target, making 4,200 due to Machine 3 downtime.
*   Quality found 80 units with paint drying issues, and the steel supply is delayed until Thursday.
*   Actions include planning extra shifts, completing a full machine check by Friday, and reporting on the paint issue by tomorrow.


And finally — **save the AI's answer to disk** so it doesn't vanish:

In [9]:
with open("summary.txt", "w", encoding="utf-8") as f:
    f.write(summary)

print("Summary saved to summary.txt 🎉")

Summary saved to summary.txt 🎉


That's the whole loop: **read a doc → ask the LLM → save the result.** You just built a document summarizer in ~15 lines.

> 📌 *Heads up:* this works great for one document that fits in the prompt. When you need to do this over **hundreds** of documents, that technique is called **RAG** — a topic for a dedicated AI course, not this Python crash course. For now, "read file → ask" gets you surprisingly far.

## 4. Saving structured data: JSON files

Plain text is fine for prose. But remember the **structured output** lesson, where the LLM gave us neat data like `{"title": ..., "calories": ...}`? To save *that* kind of data, we use **JSON files** — and Python's built-in `json` module does the heavy lifting:

- `json.dump(data, f)` — write a Python dict/list **to** a file.
- `json.load(f)` — read it **back** into a real Python dict/list.

In [ ]:
import json

recipe = {
    "title": "Yellow Dal",
    "calories": 280,
    "ingredients": ["lentils", "onion", "tomato", "cumin"]
}

# save the dict as a .json file
with open("recipe.json", "w", encoding="utf-8") as f:
    json.dump(recipe, f, indent=2)

print("recipe.json saved.")

In [ ]:
# load it back — note it returns a real dict, ready to use
with open("recipe.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)

print(type(loaded))
print(loaded["title"])
print(loaded["ingredients"])

Why bother with JSON instead of plain text? Because `json.load` hands you back a **real Python dict** — `loaded["calories"]` is instantly usable. With plain text you'd be stuck parsing strings by hand (remember the painful fence-stripping from the structured-output lesson? 😅).

**Rule of thumb:** prose for humans → `.txt`. Structured data for your code → `.json`.

## 🏋️ Exercise: Meeting notes → action items

Build a little tool that turns messy meeting notes into a saved to-do list.

1. Reuse the `meeting.txt` file from Part 3 (or write your own).
2. Read it, and ask the LLM to **extract the action items** — who needs to do what.
3. Save the result to `action_items.txt`.

**Bonus (ties everything together):** instead of plain text, ask for **structured output** (a Pydantic model with a list of `{task, owner}`), then save it with `json.dump` to `action_items.json`. Load it back and print each task. Now you've combined files + structured output into one mini-app. 🚀

In [ ]:
# Your solution here
